# 25 — Spark Execution & Lazy Evaluation

> **Sketch note:** read the visual model before the code. Predict first.

![Sketch note](../assets/sketch-notes/25-spark-execution-lazy-evaluation.svg)

## Why this exists

Large-scale systems are mechanisms for moving work, state and time under constraints. This notebook builds the mechanism before introducing a vendor.

## Learning objectives

- Build a transferable mental model.
- Predict before execution.
- Measure behavior with deterministic data.
- Break one assumption and diagnose the result.
- Connect the mechanism to production decisions.

## Prerequisites

Python basics and earlier data-engineering modules. No paid cloud service or external API is required.

## Mental model

Plan first → run later


## Capability contract

**Capability:** C06 — Streaming Lakehouse & Scale  
**Workstream:** Distributed Data Platform  
**Primary roles:** Data Engineer|Platform Engineer|Architect  
**You will prove:** Measure a scale bottleneck and choose a bounded mitigation with explicit trade-offs.


## Simulation → reality bridge

**Real-system bridge:** Apache Spark / distributed execution

**What this simulation proves:** It isolates the mechanism under study so you can predict and measure it without infrastructure noise.

**What the simulation hides:** The notebook models execution lazily with a small dependency graph. Real Spark adds JVM execution, scheduling, serialization, task retries, executors, shuffle and cluster resource behavior.

**Transfer challenge:** Reproduce the same logical plan with a real Spark runtime when available, then compare logical work with physical execution evidence.

**Production boundary:** Do not describe the toy implementation as the production technology. Before shipping, identify the real-system evidence, operational controls, and failure modes that remain untested here.


In [ ]:
from dataclasses import dataclass
@dataclass
class Op: name: str; parents: tuple[str,...]=()
plan={"read_orders":Op("read_orders"),"filter_paid":Op("filter_paid",("read_orders",)),"group_customer":Op("group_customer",("filter_paid",)),"write_metric":Op("write_metric",("group_customer",))}
print("Declared plan:", list(plan))

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Predict before you run

Write the expected behavior first.

In [ ]:
predicted=["read_orders","filter_paid","group_customer","write_metric"]; predicted

## Build

Implement or inspect the smallest mechanism that answers the question.

In [ ]:
def execution_order(plan,target):
    seen=set(); order=[]
    def visit(name):
        if name in seen:return
        for parent in plan[name].parents:visit(parent)
        seen.add(name);order.append(name)
    visit(target);return order
actual=execution_order(plan,"write_metric");print(actual)

## Measure

Turn behavior into evidence. Avoid unsupported performance claims.

In [ ]:
print("planned:",len(plan),"executed:",len(actual),"unused:",{k for k in plan if k not in actual})

## Break

Introduce one controlled failure or adverse condition.

In [ ]:
plan["count_orders"]=Op("count_orders",("read_orders",));print(execution_order(plan,"count_orders"))

## Diagnose

Explain the difference between expected and observed behavior.

In [ ]:
print("Diagnosis: lazy evaluation defers execution until an action is requested, giving an engine a complete dependency graph to reason about.")

## Production

Real systems add network cost, memory limits, retries, observability, security, schema contracts and operational ownership. This is a mental model, not a production guarantee.

## Explain it in 60–90 seconds

Explain the problem, mechanism, state transition and trade-off without naming a vendor first.

## Challenge

Change one assumption, predict the result, then connect it to the paired Acme Commerce ticket.

## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S09 — DATA-1120: Event stream backlog** → [`../work_simulations/09_acme_commerce_stream_backlog/README.md`](../work_simulations/09_acme_commerce_stream_backlog/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Takeaways

- Mechanism first, tool second.
- Predict before execution.
- Measure what can be measured.
- Break deliberately.
- Diagnose before fixing.
- Carry the mental model into engineering work.